In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_raw = pd.read_excel(
    "../data/raw/DatasetIcotMond.xlsx"
)

print(df_raw.shape)
print("ID unique:", df_raw["ID"].nunique())

In [ ]:
cols_needed = [
    "ID",
    "Age",
    "Duration (years)",
    "Gait Speed",
    "target_bin"
]

df_raw_min = df_raw[cols_needed].copy()

In [ ]:
df_raw_min["ID"] = (
    df_raw_min["ID"]
    .astype(str)
    .str.strip()
)

In [ ]:
df_raw_min["Gait Speed"] = pd.to_numeric(
    df_raw_min["Gait Speed"],
    errors="coerce"
)

In [ ]:
df_raw_min["target_bin"] = pd.to_numeric(
    df_raw_min["target_bin"],
    errors="coerce"
)

In [ ]:
df_raw_min["target_bin"].value_counts(dropna=False)

In [ ]:
df_clean = df_raw_min.dropna(
    subset=["Gait Speed", "target_bin"]
).copy()

In [ ]:
print(df_clean.shape)
print("ID unique:", df_clean["ID"].nunique())
print(df_clean["target_bin"].value_counts())

In [ ]:
df_subject = (
    df_clean
    .groupby("ID", as_index=False)
    .agg(
        Age=("Age", "first"),
        disease_duration=("Duration (years)", "first"),
        gait_speed=("Gait Speed", "mean"),
        fall=("target_bin", "first"),
        n_obs=("Gait Speed", "count")
    )
)

In [ ]:
print(df_subject.shape)
print("ID unique:", df_subject["ID"].nunique())
print(df_subject["fall"].value_counts())
print("NaN gait_speed:", df_subject["gait_speed"].isna().sum())

In [ ]:
df_subject = df_subject.copy()
df_subject.to_csv("../data/processed/subject_level_gait_speed_fall.csv", index=False)


In [ ]:
df_subject.groupby("fall")["gait_speed"].describe()

In [ ]:
df_subject["log_gait_speed"] = np.log(df_subject["gait_speed"])

In [ ]:
assert np.isfinite(df_subject["log_gait_speed"]).all()

In [ ]:
import numpy as np
import pandas as pd

# 1) Controlla dtype
print("gait_speed dtype:", df_subject["gait_speed"].dtype)

# 2) Prova conversione "hard" a float (gestisce virgole e spazi)
gs = (
    df_subject["gait_speed"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.strip()
)

gs_num = pd.to_numeric(gs, errors="coerce")

print("After coercion -> NaN count:", gs_num.isna().sum())
print("Min gait_speed:", gs_num.min())
print("Count <= 0:", (gs_num <= 0).sum())

# 3) Log trasformazione safe
log_gs = np.log(gs_num)

print("log_gait_speed: NaN count:", np.isnan(log_gs).sum())
print("log_gait_speed: inf count:", np.isinf(log_gs).sum())

# 4) Se ci sono inf o NaN, stampa le righe incriminate
bad_mask = ~np.isfinite(log_gs)
print("Bad rows:", bad_mask.sum())

if bad_mask.any():
    display(df_subject.loc[bad_mask, ["ID", "gait_speed"]].head(50))

In [ ]:
df_ana = df_subject[
    ["log_gait_speed", "Age", "disease_duration"]
].copy()

In [ ]:
for col in df_ana.columns:
    df_ana[col] = pd.to_numeric(df_ana[col], errors="coerce")

In [ ]:
print("Before drop:", df_ana.shape)

df_ana = df_ana.dropna()

print("After drop:", df_ana.shape)

In [ ]:
y = df_ana["log_gait_speed"].values

X = pd.DataFrame({
    "age_c": df_ana["Age"] - df_ana["Age"].mean(),
    "dur_c": df_ana["disease_duration"] - df_ana["disease_duration"].mean()
})

X = sm.add_constant(X)

In [ ]:
assert np.isfinite(X.values).all()
assert np.isfinite(y).all()

In [ ]:
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = df_ana["disease_duration"]
y = df_ana["log_gait_speed"]

coef = np.polyfit(x, y, 1)
y_hat = coef[0] * x + coef[1]

plt.figure(figsize=(4.5, 4))
plt.scatter(x, y, alpha=0.6)
plt.plot(x, y_hat, linewidth=2)

plt.xlabel("Disease duration (years)")
plt.ylabel("Log gait speed")
plt.title("Diffuse association between gait speed and disease duration")

plt.tight_layout()
plt.show()